In [ ]:
### Please run the file just after (model.ipynb) to get the results. ###

import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix
import seaborn as sns
from google.colab import files
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

# Professor's global plot setting: ensure grid lines are behind points/bars
plt.rcParams["axes.axisbelow"] = True

# Tracking files to download
generated_images = []

def format_model_name(name):
    '''Convert model identifier to a more readable format for plotting.'''
    label_map = {
        "random_forest": "Random Forest",
        "Random Forest": "Random Forest",
        "lightgbm": "LightGBM",
        "LightGBM (Leashed)": "LightGBM",
        "svr": "SVR",
        "Support Vector Machine": "SVM",
        "ridge": "Ridge",
        "logistic_regression": "Logistic Regression",
        "Logistic Regression": "Logistic Regression",
    }
    return label_map.get(name, name)

def save_and_track(path):
    """Saves the figure in both PNG and PDF formats."""
    plt.tight_layout()
    # Save PNG
    plt.savefig(path, dpi=400)
    generated_images.append(str(path))
    # Save PDF
    pdf_path = path.with_suffix(".pdf")
    plt.savefig(pdf_path, dpi=250)
    generated_images.append(str(pdf_path))
    plt.close()
    print(f"  -> Saved: {path.name} & {pdf_path.name}")

# =====================================================================
# PLOTTING FUNCTIONS
# =====================================================================
def create_parity_plot(data, target_col, path):
    observed = data[f"observed_{target_col}"]
    predicted = data[f"predicted_{target_col}"]

    plt.figure(figsize=(5, 4))
    plt.scatter(observed, predicted, alpha=0.75, color='#1f77b4')
    lower = min(observed.min(), predicted.min())
    upper = max(observed.max(), predicted.max())
    plt.plot([lower, upper], [lower, upper], linestyle="--", color='red')
    plt.xlabel(f"Observed {target_col}")
    plt.ylabel(f"Predicted {target_col}")
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    save_and_track(path)

def create_residual_plot(data, target_col, path):
    predicted = data[f"predicted_{target_col}"]
    residual = data[f"observed_{target_col}"] - data[f"predicted_{target_col}"]

    plt.figure(figsize=(5, 4))
    plt.scatter(predicted, residual, alpha=0.75, color='#1f77b4')
    plt.axhline(0, linestyle="--", color='red')
    plt.xlabel(f"Predicted {target_col}")
    plt.ylabel("Residual (observed - predicted)")
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    save_and_track(path)

def create_metric_bar_plot(metrics_df, metric, path):
    data = metrics_df.copy()

    # Sort descending for "higher is better" metrics, ascending for errors
    if metric in ["r2", "accuracy", "f1", "precision", "recall"]:
        data = data.sort_values(metric, ascending=False)
    else:
        data = data.sort_values(metric, ascending=True)

    labels = [format_model_name(m) for m in data["model"]]
    values = data[metric].values

    plt.figure(figsize=(4.5, 3.5))
    bars = plt.bar(labels, values, color="#4c72b0")
    plt.ylim(0, max(values) * 1.25)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel(metric.upper())
    plt.grid(axis="y", linestyle="--", alpha=0.7)

    # Add exact values on top of bars
    for bar, val in zip(bars, values):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.01,
            f"{val:.3f}",
            ha="center",
            va="bottom",
        )
    save_and_track(path)

def create_feature_importance_plot(data, path):
    data = data.copy()
    data = data.head(10).sort_values("importance", ascending=True)

    plt.figure(figsize=(6, 3.5))
    bars = plt.barh(data["feature"], data["importance"], color="#55a868")
    plt.xlabel("Importance")
    plt.xlim(0, data["importance"].max() * 1.25)

    for bar, val in zip(bars, data["importance"].values):
        plt.text(
            bar.get_width() * 1.01,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}",
            va="center",
        )
    save_and_track(path)

def create_confusion_matrix_plot(predictions_df, target_col, path):
    y_true = predictions_df[f"observed_{target_col}"]
    y_pred = predictions_df[f"predicted_{target_col}"]
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(4.5, 3.5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Shallow', 'Deep'], yticklabels=['Shallow', 'Deep'])
    plt.xlabel("Predicted Foundation Type")
    plt.ylabel("Observed Foundation Type")
    save_and_track(path)

# =====================================================================
# MAIN EXECUTION
# =====================================================================
def run_all_visualizations():
    figures_dir = Path("output_figures")
    figures_dir.mkdir(parents=True, exist_ok=True)

    print("========================================================")
    print("1. GENERATING CLASSIFICATION PLOTS")
    print("========================================================")
    class_metrics_path = "output_accuracy_classification_model_comparison_metrics.csv"
    class_preds_path = "output_accuracy_classification_all_model_predictions.csv"

    if os.path.exists(class_metrics_path) and os.path.exists(class_preds_path):
        c_metrics = pd.read_csv(class_metrics_path)
        c_preds = pd.read_csv(class_preds_path)

        create_metric_bar_plot(c_metrics, "accuracy", figures_dir / "class_model_comparison_accuracy.png")
        create_metric_bar_plot(c_metrics, "f1", figures_dir / "class_model_comparison_f1.png")

        best_c_model = c_metrics.loc[0, "model"]
        best_c_preds = c_preds[c_preds["model"] == best_c_model]
        create_confusion_matrix_plot(best_c_preds, "Foundation_Type", figures_dir / "class_confusion_matrix.png")

        for model_suffix in ["random", "lightgbm"]:
            fi_path = f"output_accuracy_classification_feature_importance_{model_suffix}.csv"
            if os.path.exists(fi_path):
                imp = pd.read_csv(fi_path)
                create_feature_importance_plot(imp, figures_dir / f"class_feature_importance_{model_suffix}.png")
    else:
        print("❌ Classification CSVs not found. Skipping classification plots.")

    print("\n========================================================")
    print("2. GENERATING PREDICTION PLOTS")
    print("========================================================")
    pred_metrics_path = "output_accuracy_prediction_model_comparison_metrics.csv"
    pred_preds_path = "output_accuracy_prediction_best_model_predictions.csv"

    if os.path.exists(pred_metrics_path) and os.path.exists(pred_preds_path):
        p_metrics = pd.read_csv(pred_metrics_path)
        p_preds = pd.read_csv(pred_preds_path)

        create_metric_bar_plot(p_metrics, "rmse", figures_dir / "pred_model_comparison_rmse.png")
        create_metric_bar_plot(p_metrics, "r2", figures_dir / "pred_model_comparison_r2.png")

        best_p_model = p_metrics.loc[0, "model"]
        clean_model_name = format_model_name(best_p_model).replace(' ', '_').lower()
        create_parity_plot(p_preds, "Pier total Depth (ft)", figures_dir / f"pred_parity_plot_{clean_model_name}.png")
        create_residual_plot(p_preds, "Pier total Depth (ft)", figures_dir / f"pred_residual_plot_{clean_model_name}.png")

        for model_suffix in ["random_forest", "lightgbm"]:
            fi_path = f"output_accuracy_prediction_feature_importance_{model_suffix}.csv"
            if os.path.exists(fi_path):
                imp = pd.read_csv(fi_path)
                create_feature_importance_plot(imp, figures_dir / f"pred_feature_importance_{model_suffix}.png")
    else:
        print("❌ Prediction CSVs not found. Skipping prediction plots.")

    print("\n========================================================")
    print("3. DOWNLOADING FIGURES")
    print("========================================================")
    if generated_images:
        print(f"Initiating downloads for {len(generated_images)} files...")
        for file_path in generated_images:
            try:
                files.download(file_path)
            except Exception as e:
                print(f"Could not download {file_path}. Error: {e}")
    else:
        print("No images were generated to download.")

if __name__ == "__main__":
    run_all_visualizations()